In [ ]:
import pandas as pd
train = pd.read_csv("train.csv")


In [ ]:
#1
label_map = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}

train["label"] = train["answer"].map(label_map)
print(train.loc[150, "label"])

2


In [ ]:
#2
formatted_input = str(train.loc[0, "prompt"]) + " [SEP] " + str(train.loc[0, "B"])
print(formatted_input)
print("Character Length:", len(formatted_input))

Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. [SEP] Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.
Character Length: 407


In [ ]:
#3
from transformers import AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Create the five prompt-option pairs for row 0
row = train.loc[0]

choices = [
    str(row["prompt"]) + " [SEP] " + str(row["A"]),
    str(row["prompt"]) + " [SEP] " + str(row["B"]),
    str(row["prompt"]) + " [SEP] " + str(row["C"]),
    str(row["prompt"]) + " [SEP] " + str(row["D"]),
    str(row["prompt"]) + " [SEP] " + str(row["E"]),
]

encoding = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

input_ids = encoding["input_ids"].unsqueeze(0)

print("Shape:", input_ids.shape)
print("Second Dimension:", input_ids.shape[1])

Shape: torch.Size([1, 5, 128])
Second Dimension: 5


In [ ]:
#4
batch_input_ids = []

for i in range(16):
    row = train.loc[i]

    choices = [
        str(row["prompt"]) + " [SEP] " + str(row["A"]),
        str(row["prompt"]) + " [SEP] " + str(row["B"]),
        str(row["prompt"]) + " [SEP] " + str(row["C"]),
        str(row["prompt"]) + " [SEP] " + str(row["D"]),
        str(row["prompt"]) + " [SEP] " + str(row["E"]),
    ]

    encoding = tokenizer(
        choices,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    batch_input_ids.append(encoding["input_ids"])

batch_input_ids = torch.stack(batch_input_ids)
print("Shape:", batch_input_ids.shape)
total_positions = batch_input_ids.numel()
print("Total token positions:", total_positions)

Shape: torch.Size([16, 5, 128])
Total token positions: 10240


In [ ]:
#5
from transformers import AutoModelForMultipleChoice
mc_model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

row = train.loc[0]

choices = [
    str(row["prompt"]) + " [SEP] " + str(row["A"]),
    str(row["prompt"]) + " [SEP] " + str(row["B"]),
    str(row["prompt"]) + " [SEP] " + str(row["C"]),
    str(row["prompt"]) + " [SEP] " + str(row["D"]),
    str(row["prompt"]) + " [SEP] " + str(row["E"]),
]

encoding = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

inputs = {k: v.unsqueeze(0) for k, v in encoding.items()}

with torch.no_grad():
    outputs = mc_model(**inputs)

print("Logits shape:", outputs.logits.shape)
print("Logits:", outputs.logits)
print("Number of logits:", outputs.logits.shape[1])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Logits shape: torch.Size([1, 5])
Logits: tensor([[0.1431, 0.1120, 0.0669, 0.0977, 0.0805]])
Number of logits: 5


In [ ]:
#6
label = torch.tensor([train.loc[0, "label"]])
with torch.no_grad():
    outputs = mc_model(**inputs, labels=label)

print("Loss:", outputs.loss)
print("Loss shape:", outputs.loss.shape)
print("Number of dimensions:", outputs.loss.dim())

Loss: tensor(1.5978)
Loss shape: torch.Size([])
Number of dimensions: 0


In [ ]:
#7
!pip -q install peft

In [ ]:
!pip install -q --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 47.9 MB/s eta 0:00:00


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

base_model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)

lora_model = get_peft_model(base_model, lora_config)

trainable_params = sum(
    p.numel() for p in lora_model.parameters() if p.requires_grad
)

print("Trainable Parameters:", trainable_params)

lora_model.print_trainable_parameters()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable Parameters: 295681
trainable params: 295,681 || all params: 109,778,690 || trainable%: 0.2693


In [ ]:
#8
from datasets import Dataset
train_100 = train.iloc[:100].copy()

def preprocess(example):
    choices = [
        str(example["prompt"]) + " [SEP] " + str(example["A"]),
        str(example["prompt"]) + " [SEP] " + str(example["B"]),
        str(example["prompt"]) + " [SEP] " + str(example["C"]),
        str(example["prompt"]) + " [SEP] " + str(example["D"]),
        str(example["prompt"]) + " [SEP] " + str(example["E"]),
    ]

    encoding = tokenizer(
        choices,
        padding="max_length",
        truncation=True,
        max_length=128,
    )

    return {
        "input_ids": encoding["input_ids"],
        "attention_mask": encoding["attention_mask"],
        "labels": int(example["label"])
    }

hf_dataset = Dataset.from_pandas(train_100)

hf_dataset = hf_dataset.map(preprocess)

print("input_ids shape:", (
    len(hf_dataset[0]["input_ids"]),
    len(hf_dataset[0]["input_ids"][0])
))
print("Number of tokenized choices:", len(hf_dataset[0]["input_ids"]))

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

input_ids shape: (5, 128)
Number of tokenized choices: 5


In [ ]:
#9
from datasets import Dataset
from transformers import (
    AutoModelForMultipleChoice,
    Trainer,
    TrainingArguments,
    DefaultDataCollator,
)
from peft import LoraConfig, get_peft_model, TaskType

train_32 = train.iloc[:32].copy()

def preprocess_train(example):
    choices = [
        str(example["prompt"]) + " [SEP] " + str(example["A"]),
        str(example["prompt"]) + " [SEP] " + str(example["B"]),
        str(example["prompt"]) + " [SEP] " + str(example["C"]),
        str(example["prompt"]) + " [SEP] " + str(example["D"]),
        str(example["prompt"]) + " [SEP] " + str(example["E"]),
    ]

    enc = tokenizer(
        choices,
        padding="max_length",
        truncation=True,
        max_length=64,
    )

    return {
        "input_ids": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "labels": int(example["label"]),
    }

dataset = Dataset.from_pandas(train_32)
dataset = dataset.map(preprocess_train, remove_columns=dataset.column_names)

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

In [ ]:
base_model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)

model = get_peft_model(base_model, lora_config)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
training_args = TrainingArguments(
    output_dir="./mcq_lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=DefaultDataCollator(),
)

trainer.train()

print("Final global_step:", trainer.state.global_step)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,1.623632
2,1.605257
3,1.585512
4,1.581583


Final global_step: 4


In [ ]:
import torch

row = train.loc[0]

choices = [
    str(row["prompt"]) + " [SEP] " + str(row["A"]),
    str(row["prompt"]) + " [SEP] " + str(row["B"]),
    str(row["prompt"]) + " [SEP] " + str(row["C"]),
    str(row["prompt"]) + " [SEP] " + str(row["D"]),
    str(row["prompt"]) + " [SEP] " + str(row["E"]),
]

encoding = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=64,      # same as Q9
    return_tensors="pt"
)

inputs = {k: v.unsqueeze(0) for k, v in encoding.items()}

model.eval()
with torch.no_grad():
    outputs = model(**inputs)

probs = torch.softmax(outputs.logits, dim=-1)

print("Probabilities:", probs)
print("Probability of Option E:", round(probs[0, 4].item(), 4))

Probabilities: tensor([[0.2068, 0.2022, 0.1977, 0.1968, 0.1965]])
Probability of Option E: 0.1965
